In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_agreement_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "AgreementId string, SourceSystem string,"
        #     "PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, BatchId string, AgreementKey string,"
        #     "AgreementStatusCode string, AgreementStatus string, IssueDate date,"
        #     "ExpiredDate date, SignedDate date, Currency string,"
        #     "LimitAmount decimal(19,2), LimitUtilizedAmount decimal(19,2),"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Agreement"

        # ================== Create Delta Table ==================

        # logging.info("Creating Delta Table...")
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")


        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "qca",
                )        
        else:
            logging.info("Tables do not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "portfolio_id_qca"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "CASE_CODE",
            "CASE_CODE_PRIMARY",
            "CASE_S_ID",
            "CASE_CONTRACT_ACCOUNT_KEY",
            "CASE_EXP_DATE",
            "CASE_LEGAL_CONTRACT",
            "CASE_ACCOUNT_FOR_SEARCH",
            "businessdate",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )

        # ================== FS_CASES_SNAP ==================

        FS_CASES_SNAP_extra_filters = None

        if FS_CASES_SNAP_extra_filters is not None:
            FS_CASES_SNAP_extra_filters.append(date_filter)
            FS_CASES_SNAP_final_filters = " AND ".join(FS_CASES_SNAP_extra_filters)
        else:
            FS_CASES_SNAP_final_filters = date_filter

        FS_CASES_SNAP_selected_fields = [
            "CASE_CODE",
            "CAEX_CUSTOM_DATETIME_03",
            "SPET_CODE",
            "CASE_SOURCE",
        ]

        FS_CASES_SNAP = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_CASES_SNAP",
            FS_CASES_SNAP_final_filters,
            FS_CASES_SNAP_selected_fields,
        )

        # ================== Join conditions ==================

        FS_CASES_SNAP_dim_cases_static_join_condition = (
            FS_CASES_SNAP["CASE_CODE"] == dim_cases_static["CASE_CODE"]
        ) & (dim_cases_static["CASE_CODE"] == dim_cases_static["CASE_CODE_PRIMARY"])

        FS_CASES_SNAP_portfolio_join_condition = (
            FS_CASES_SNAP["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        result_df_qca = FS_CASES_SNAP.join(
            dim_cases_static,
            FS_CASES_SNAP_dim_cases_static_join_condition,
            "inner",
        ).join(
            portfolio,
            FS_CASES_SNAP_portfolio_join_condition,
            "left",
         )

        result_df_qca = result_df_qca.withColumn(
            "CASE_CONTRACT_ACCOUNT_KEY_new",
            when(
                (FS_CASES_SNAP.CASE_SOURCE).like("C%"),
                dim_cases_static.CASE_CONTRACT_ACCOUNT_KEY,
            ).otherwise(dim_cases_static.CASE_LEGAL_CONTRACT),
        )

        dim_cases_static_primary_key_fields = [
            result_df_qca.CASE_CONTRACT_ACCOUNT_KEY_new,
            lit("qca"),
        ]

        result_df_qca = result_df_qca.withColumn(
            "AgreementId",
            sha2(concat_ws(";", *dim_cases_static_primary_key_fields), 256),
        )
        # .withColumn("SignedDate",
        #     when(
        #         (FS_CASES_SNAP.CAEX_CUSTOM_DATETIME_03 == "1753-01-01 00:00:00.000")
        #         | (FS_CASES_SNAP.CAEX_CUSTOM_DATETIME_03 == "1900-01-01 00:00:00.000")
        #         | (FS_CASES_SNAP.CAEX_CUSTOM_DATETIME_03.isNull()),
        #         lit(None)
        #     ).otherwise(FS_CASES_SNAP.CAEX_CUSTOM_DATETIME_03)
        #     )

        # ================== Selection of Final Columns ==================

        result_df_qca_final = result_df_qca.select(
            result_df_qca.AgreementId,
            lit("qca").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            result_df_qca.CASE_CONTRACT_ACCOUNT_KEY_new.alias("AgreementKey"),
            lit(None).alias("AgreementStatusCode"),
            lit(None).alias("AgreementStatus"),
            lit(None).alias("IssueDate"),
            lit(None).cast("date").alias("ExpiredDate"),
            # result_df_qca.SignedDate.cast("date").alias("SignedDate"),
            lit(None).cast("date").alias("SignedDate"),
            lit(None).alias("Currency"),
            lit(None).alias("LimitAmount"),
            lit(None).alias("LimitUtilizedAmount"),
            result_df_qca.businessdate.alias("BusinessDate"),
        ).distinct()

        # ================== Append ==================

        result_df_qca_final.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")



In [0]:
load_curated_agreement_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)